# RHONN Training for Differential Drive Mobile Robot

## 🎯 **Objective**
This notebook demonstrates **Recurrent High-Order Neural Networks (RHONN)** training for a differential drive mobile robot using three different filtering approaches:
- **Extended Kalman Filter (EKF)**  
- **Unscented Kalman Filter (UKF)**
- **Particle Filter (PF)**

## 📋 **What This Notebook Does**
1. **Simulates a realistic differential drive robot** with friction, wheel slip, and terrain disturbances
2. **Trains RHONN models** to learn the robot's nonlinear dynamics using different filtering algorithms
3. **Optimizes hyperparameters** using Differential Evolution for EKF and UKF (PF parameters are set manually)
4. **Compares performance** between the three filtering approaches using MSE metrics
5. **Visualizes results** with trajectory plots and error analysis

## 🏗️ **Notebook Structure**
- **Section 1**: Robot dynamics and simulation environment
- **Section 2**: RHONN neural network architecture and prediction functions  
- **Section 3**: Filter implementations (EKF, UKF, PF)
- **Section 4**: Differential Evolution hyperparameter optimization
- **Section 5**: Main simulation and training
- **Section 6**: Results analysis and visualization

## 📚 **Required Libraries**

We need the following libraries for this simulation:
- **NumPy**: For numerical computations and array operations
- **Plotly**: For interactive 3D trajectory visualization and error plots

In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
# ============================================================
# 🎲 VARIABLE SEED: Genera resultados diferentes pero rastreables
# ============================================================
import numpy as np
import time

# Genera una semilla basada en el tiempo actual (microsegundos)
# Esto asegura que cada ejecución tenga resultados diferentes
RANDOM_SEED = int((time.time() * 1000000) % 100000)  # Semilla entre 0-99999
np.random.seed(RANDOM_SEED)

# Mensaje prominente para rastrear el rendimiento
print("🎲" + "="*60)
print(f"🎯 SEMILLA ACTUAL: {RANDOM_SEED}")
print("   ⚡ ANOTA ESTA SEMILLA SI OBTIENES BUENOS RESULTADOS")
print("   🔄 Para reproducir: cambia línea 8 a RANDOM_SEED = {0}".format(RANDOM_SEED))
print("="*62)

🎲============================================================
🎯 SEMILLA ACTUAL: 58865
   ⚡ ANOTA ESTA SEMILLA SI OBTIENES BUENOS RESULTADOS
   🔄 Para reproducir: cambia línea 8 a RANDOM_SEED = 58865


# 🤖 **Section 1: Differential Drive Mobile Robot Model**

## **Robot Dynamics Overview**

A differential drive mobile robot has two independently controlled wheels. The robot state is represented by:
- **x**: Position along x-axis (meters)  
- **y**: Position along y-axis (meters)
- **θ (theta)**: Orientation angle (radians)

## **Control Inputs**
- **v_l**: Left wheel velocity (m/s)
- **v_r**: Right wheel velocity (m/s)

## **Mathematical Model**
The robot kinematics follow:
```
dx/dt = v * cos(θ)
dy/dt = v * sin(θ)  
dθ/dt = ω

where:
v = (v_r + v_l) / 2    # Linear velocity
ω = (v_r - v_l) / L    # Angular velocity  
L = wheelbase distance
```

## **Realistic Effects Modeled**
- ⚙️ **Wheel Slip**: Random slip factors affecting actual wheel velocities
- 🔥 **Friction**: Velocity-dependent friction reducing motion
- 🌍 **Terrain Disturbances**: Ground roughness affecting movement  
- 📡 **Sensor Noise**: Measurement noise and bias in position/orientation sensors

In [3]:
# ============================================================
# 1) True nonlinear system (Differential Drive Mobile Robot)
# ============================================================
def plant_dynamics(x, u, L=0.5, friction_coeff=0.1, slip_factor=0.05):
    """
    Continuous dynamics for differential drive mobile robot: x = [x_pos, y_pos, theta]. 
    Returns x_dot.
    
    The differential drive robot equations:
    dx/dt = v * cos(θ) 
    dy/dt = v * sin(θ)
    dθ/dt = ω
    
    where:
    v = (v_r + v_l) / 2  (linear velocity)
    ω = (v_r - v_l) / L  (angular velocity)
    L = wheelbase distance
    """
    x_pos, y_pos, theta = x
    v_l, v_r = u  # left and right wheel velocities
    
    # Add realistic wheel slip effects
    v_l_actual = v_l * (1 - slip_factor * np.random.randn())
    v_r_actual = v_r * (1 - slip_factor * np.random.randn())
    
    # Compute linear and angular velocities
    v = (v_r_actual + v_l_actual) / 2.0
    omega = (v_r_actual - v_l_actual) / L
    
    # Add friction effects (velocity-dependent)
    v_friction = v * (1 - friction_coeff * np.abs(v))
    omega_friction = omega * (1 - friction_coeff * np.abs(omega))
    
    # Robot kinematics
    x_dot = v_friction * np.cos(theta)
    y_dot = v_friction * np.sin(theta)
    theta_dot = omega_friction
    
    return np.array([x_dot, y_dot, theta_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='mixed', process_noise_std=0.05, 
          terrain_roughness=0.02, sensor_bias=[0.0, 0.0, 0.0]):
    """
    One Euler step of the discrete plant with realistic mobile robot disturbances.
    
    Args:
        x_k: current state [x, y, theta]
        u_k: control input [v_left, v_right] 
        dt: time step
        process_noise_type: type of noise ('mixed', 'gaussian', 'laplacian')
        process_noise_std: standard deviation of process noise
        terrain_roughness: terrain-induced disturbances
        sensor_bias: systematic biases in measurements
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Realistic mobile robot disturbances
    
    # 1. Terrain-induced disturbances (position-dependent)
    terrain_noise = terrain_roughness * np.array([
        np.sin(0.5 * x_kp1[0]) * np.random.randn(),  # x-direction terrain variation
        np.cos(0.3 * x_kp1[1]) * np.random.randn(),  # y-direction terrain variation  
        0.1 * np.sin(x_kp1[2]) * np.random.randn()   # angular disturbance from terrain
    ])
    
    # 2. Velocity-dependent noise (increases with speed)
    velocity_magnitude = np.linalg.norm(u_k)
    velocity_noise_factor = 1 + 0.2 * velocity_magnitude
    
    # 3. Mixed process noise (combination of different noise types)
    if process_noise_type == 'mixed':
        # Gaussian component (main noise)
        gaussian_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
        # Impulse noise (occasional large disturbances)
        impulse_prob = 0.02  # 2% chance of impulse noise
        impulse_noise = np.zeros_like(x_kp1)
        if np.random.rand() < impulse_prob:
            impulse_noise = np.random.normal(0, process_noise_std * 5, size=x_kp1.shape)
        # Laplacian component (heavy-tailed noise)
        laplacian_noise = np.random.laplace(0, process_noise_std * 0.3, size=x_kp1.shape)
        
        total_noise = gaussian_noise + impulse_noise + laplacian_noise
    elif process_noise_type == 'laplacian':
        total_noise = np.random.laplace(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std * velocity_noise_factor
        total_noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        total_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    
    # 4. Systematic biases (drift, calibration errors)
    bias_noise = np.array(sensor_bias) * dt
    
    # 5. Wheel encoder quantization effects
    encoder_resolution = 0.001  # 1mm resolution
    quantization_noise = encoder_resolution * (np.random.rand(3) - 0.5)
    
    # Combine all disturbances
    x_kp1 += terrain_noise + total_noise + bias_noise + quantization_noise
    
    # 6. Angle wrapping for realistic behavior
    x_kp1[2] = np.arctan2(np.sin(x_kp1[2]), np.cos(x_kp1[2]))  # wrap angle to [-π, π]
    
    return x_kp1

def generate_realistic_trajectory(t, trajectory_type='straight'):
    """
    Generate realistic control inputs for mobile robot.
    
    Args:
        t: time value
        trajectory_type: 'straight', 'circle', 'figure8', 'mixed', 'obstacle_avoidance'
    
    Returns:
        u: [v_left, v_right] wheel velocities
    """
    if trajectory_type == 'straight':
        # Straight line with small variations
        v_base = 1.0 + 0.2 * np.sin(0.5 * t)
        return np.array([v_base, v_base])
    
    elif trajectory_type == 'circle':
        # Circular motion
        v_l = 1.0 + 0.1 * np.sin(t)
        v_r = 1.5 + 0.1 * np.cos(t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'figure8':
        # Figure-8 pattern
        v_l = 1.0 + 0.8 * np.sin(0.5 * t)
        v_r = 1.0 - 0.8 * np.sin(0.5 * t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'obstacle_avoidance':
        # Obstacle avoidance maneuvers
        base_speed = 1.2
        avoidance_maneuver = 0.5 * np.sin(2 * t) * np.exp(-0.1 * t)
        v_l = base_speed + avoidance_maneuver
        v_r = base_speed - avoidance_maneuver
        return np.array([v_l, v_r])
    
    else:  # 'mixed' - combination of different behaviors
        # Mixed trajectory with different phases
        phase = (t % 20.0) / 20.0  # 20-second cycles
        
        if phase < 0.3:  # Straight motion
            v_base = 1.5
            return np.array([v_base, v_base])
        elif phase < 0.6:  # Turning
            v_l = 0.8
            v_r = 1.8
            return np.array([v_l, v_r])
        elif phase < 0.8:  # Reverse
            v_base = -0.5
            return np.array([v_base, v_base])
        else:  # Complex maneuver
            v_l = 1.0 + 0.5 * np.sin(10 * t)
            v_r = 1.0 + 0.5 * np.cos(10 * t)
            return np.array([v_l, v_r])

# 🧠 **Section 2: RHONN Neural Network Architecture**

## **What is a RHONN?**
A **Recurrent High-Order Neural Network (RHONN)** is a special type of neural network that:
- 🔄 Uses **recurrent connections** to handle temporal dependencies  
- 🎯 Employs **high-order terms** (products of inputs) to capture complex nonlinear relationships
- ⚡ Is particularly effective for **nonlinear system identification**

## **RHONN Architecture for Mobile Robot**

For our differential drive robot with state **x = [x_pos, y_pos, θ]** and controls **u = [v_l, v_r]**, we use:

### **Feature Vector Construction**
The RHONN uses an **extended feature vector z** that includes:
1. **Sigmoid transformations**: S(x), S(y), S(θ) 
2. **Cross-product terms**: S(x)S(y), S(x)S(θ), S(y)S(θ)
3. **Quadratic terms**: S(x)², S(y)², S(θ)²
4. **Trigonometric terms**: cos(θ), sin(θ) *(crucial for robot orientation)*
5. **Control terms**: S(v_l), S(v_r), S(v_l)S(v_r)
6. **Linear terms**: x, y *(for position tracking)*
7. **Bias term**: 1

### **Network Structure**
- **3 neurons** (one for each state: x, y, θ)
- **17 features** per neuron (comprehensive feature set)
- **Series-parallel identification**: Uses true state for feature construction

### **Prediction Equation**
Each neuron computes: **ŝᵢ(k+1) = wᵢᵀ × z(k)**

Where:
- **wᵢ**: Weight vector for neuron i
- **z(k)**: Feature vector at time k  
- **ŝᵢ(k+1)**: Predicted state component i at time k+1

In [4]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    # z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input=None):
    """
    Features for a 3-state mobile robot system with control inputs:
    x = [x_pos, y_pos, theta], u = [v_left, v_right]
    
    z = [S(x), S(y), S(θ), S(x)S(y), S(x)S(θ), S(y)S(θ), 
         S(x)^2, S(y)^2, S(θ)^2, cos(θ), sin(θ), 
         S(v_l), S(v_r), S(v_l)S(v_r), x, y, 1]
    """
    s_x = sigmoidal(x_est[0])     # x position
    s_y = sigmoidal(x_est[1])     # y position  
    s_theta = sigmoidal(x_est[2]) # orientation
    
    # Basic features
    features = [
        s_x, s_y, s_theta,                    # Individual sigmoid terms
        s_x*s_y, s_x*s_theta, s_y*s_theta,   # Cross terms
        s_x**2, s_y**2, s_theta**2,          # Quadratic terms
        np.cos(x_est[2]), np.sin(x_est[2]),  # Trigonometric terms (important for robot)
    ]
    
    # Add control input features if available
    if u_input is not None and len(u_input) >= 2:
        s_vl = sigmoidal(u_input[0])  # left wheel velocity
        s_vr = sigmoidal(u_input[1])  # right wheel velocity
        features.extend([
            s_vl, s_vr,                       # Control sigmoid terms
            s_vl * s_vr,                      # Control cross term
        ])
    else:
        # Add zero placeholders if no control input
        features.extend([0.0, 0.0, 0.0])
    
    # Add direct state terms and bias
    features.extend([
        x_est[0], x_est[1],                   # Direct position terms
        1.0                                   # Bias term
    ])
    
    return np.array(features)


def RHONN_predict(x_state_for_z, w_neuron, u_input=None):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z, u_input)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# 🔧 **Section 3: Filter-Based RHONN Training Algorithms**

## **Training Philosophy**
Instead of traditional backpropagation, we use **filtering algorithms** to train RHONN weights. Each filter treats the **weight vector of each neuron as a state** to be estimated.

## **Training Approach: Series-Parallel Identification**
- **Input at time k**: True robot state x(k) and control u(k)
- **Target at time k+1**: True robot state x(k+1) 
- **Prediction**: ŝ(k+1) = RHONN(x(k), u(k))
- **Error**: e(k+1) = x(k+1) - ŝ(k+1)

## **Three Filter Implementations**

### 🎯 **1. Extended Kalman Filter (EKF)**
- **Treats weights as states**: w ~ N(μ, P)
- **Linear measurement model**: y = H*w + noise
- **Handles nonlinearity** through first-order linearization
- **Pros**: Computationally efficient, well-established
- **Cons**: Assumes Gaussian distributions, linearization errors

### 🎯 **2. Unscented Kalman Filter (UKF)** 
- **Uses sigma points** to capture nonlinear transformations
- **No Jacobian calculation** required (unlike EKF)
- **Better nonlinearity handling** through deterministic sampling
- **Pros**: More accurate for nonlinear systems than EKF
- **Cons**: Higher computational cost than EKF

### 🎯 **3. Particle Filter (PF)**
- **Uses particle cloud** to represent weight distributions
- **No Gaussian assumption** - can handle any distribution
- **Stratified resampling** for reduced variance
- **Pros**: Most flexible, handles multi-modal distributions
- **Cons**: Highest computational cost, needs tuning

## **Hyperparameter Optimization**
- **EKF & UKF**: Parameters optimized using **Differential Evolution**
- **PF**: Parameters set **manually** for direct control

In [5]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [6]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=None, R_std=None, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Handle Q_std (process noise) - can be scalar or list/array per neuron
        if Q_std is None:
            self.Q_std = [0.05] * num_neurons
        elif isinstance(Q_std, (int, float)):
            self.Q_std = [Q_std] * num_neurons
        else:
            self.Q_std = list(Q_std) if len(Q_std) == num_neurons else [0.05] * num_neurons
            
        # Handle R_std (measurement noise) - can be scalar or list/array per neuron  
        if R_std is None:
            self.R_std = [0.1] * num_neurons
        elif isinstance(R_std, (int, float)):
            self.R_std = [R_std] * num_neurons
        else:
            self.R_std = list(R_std) if len(R_std) == num_neurons else [0.1] * num_neurons
            
        # Compute R_var for each neuron
        self.R_var = [r**2 for r in self.R_std]
        
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        # Initialize global weight estimates first
        self.weights = []
        if initial_weights is not None:
            self.weights = [np.copy(w) for w in initial_weights]
        else:
            self.weights = [np.random.randn(num_weights_per_neuron) * 0.01 for _ in range(num_neurons)]

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                # Adaptive initialization variance based on weight magnitudes
                weight_magnitude = np.std(base) if np.std(base) > 0 else 1.0
                init_std = max(0.01, min(0.1, weight_magnitude * 0.5))  # Adaptive but bounded
            else:
                base = self.weights[i]
                init_std = 0.05
            
            # Better initialization: base + controlled noise
            particles_i = base[np.newaxis, :] + np.random.randn(n_particles, num_weights_per_neuron) * init_std
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        """Effective Sample Size calculation with improved numerical stability."""
        w_norm = w / (np.sum(w) + 1e-15)
        return 1.0 / (np.sum(w_norm**2) + 1e-15)

    def _resample_stratified(self, neuron_index):
        """Stratified resampling (reduces variance compared to systematic/multinomial)"""
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w_norm = w / (np.sum(w) + 1e-15)
        N = len(w_norm)
        cdf = np.cumsum(w_norm)
        
        # Stratified positions: divide [0,1] into N strata
        positions = (np.random.rand(N) + np.arange(N)) / N
        
        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N and j < N:
            if positions[i] <= cdf[j]:
                indexes[i] = j
                i += 1
            else:
                j += 1
        
        # Handle any remaining indices
        while i < N:
            indexes[i] = N - 1
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : filter's own estimate at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        u_input: control input at time k (for mobile robot)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)
        z = construct_z_vector(x_state_for_z, u_input)  # (num_features,)

        # 1) Predict: random walk on weights with state-specific Q_std
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std[i]

        # 2) Update: importance weights with improved Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Improved log-likelihood calculation with better numerical stability
            var_robust = max(self.R_var[i], 1e-6)  # Avoid division by very small numbers
            ll = -0.5 * (innov**2) / var_robust - 0.5 * np.log(2 * np.pi * var_robust)
            
            # Normalize for numerical stability
            ll_max = np.max(ll)
            ll_normalized = ll - ll_max
            like = np.exp(np.clip(ll_normalized, -20, 0))  # Clip to avoid underflow

            # Update weights with better safeguards
            self.weights_pf[i] *= (like + 1e-15)
            w_sum = np.sum(self.weights_pf[i])
            
            if w_sum < 1e-15:
                # Complete weight collapse - reinitialize uniformly
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= w_sum

            # 3) Resample if ESS is low (using improved stratified resampling)
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_stratified(i)

        # 4) Update global weight estimates (weighted mean of particles for consistency)
        if not hasattr(self, 'weights'):
            self.weights = []
        
        # Ensure we have the right number of weight vectors
        while len(self.weights) < self.num_neurons:
            self.weights.append(np.zeros(self.num_weights_per_neuron))
            
        for i in range(self.num_neurons):
            w_norm = self.weights_pf[i] / (np.sum(self.weights_pf[i]) + 1e-15)
            self.weights[i] = np.sum(w_norm[:, np.newaxis] * self.particles[i], axis=0)

    def get_estimate(self):
        """Return current weight estimates (maintained consistently with particles)."""
        if hasattr(self, 'weights') and len(self.weights) == self.num_neurons:
            return self.weights
        else:
            # Fallback to simple mean if weights not properly maintained
            return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

    def get_parameters_info(self):
        """Return comprehensive information about the PF parameters and state for each neuron."""
        state_names = ['x', 'y', 'theta']
        info = {}
        for i in range(self.num_neurons):
            name = state_names[i] if i < len(state_names) else f'state_{i}'
            current_ess = self._ess(self.weights_pf[i]) if hasattr(self, 'weights_pf') else 'N/A'
            info[name] = {
                'Q_std': self.Q_std[i],
                'R_std': self.R_std[i], 
                'R_var': self.R_var[i],
                'n_particles': self.n_particles,
                'ess_threshold': self.ess_threshold,
                'current_ess': current_ess,
                'ess_ratio': current_ess / self.n_particles if isinstance(current_ess, (int, float)) else 'N/A'
            }
        return info

In [7]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

## 🔧 Parameter Optimization via Differential Evolution (DE)

We'll optimize filter hyperparameters using **Differential Evolution**, a population-based global optimizer well-suited for noisy, non-convex error landscapes.

### Why Differential Evolution?
- Gradient-free (robust to non-differentiable effects from resampling / stochastic noise)
- Maintains population diversity (helps avoid premature convergence)
- Simple control (mutation factor F, crossover rate CR)

### Filters & Tunable Parameters
We optimize (separately) the following continuous parameters:

| Filter | Parameters (vector order) | Bounds |
|--------|---------------------------|--------|
| EKF | [Q_init, R_init, P_init, eta] | [1e-6–1e-2, 1e-5–1e-1, 0.1–10, 0.1–1.2] |
| UKF | [Q_init, R_init, P_init, eta, alpha] | same first 4 + alpha: 1e-4–0.5 |
| PF  | [Q_x, Q_y, Q_theta, R_x, R_y, R_theta, ess_ratio] | Q/R: 1e-3–1.0, ess_ratio: 0.3–0.9 |

Number of particles (500) is fixed as requested.

### Objective Function
For each candidate θ:
1. Run a shortened simulation horizon (e.g. reduced n_steps) for speed
2. Compute total MSE = Σ(MSE_x + MSE_y + MSE_theta)
3. Return total MSE (lower is better)

### Computational Strategy
- Use moderate population size (pop = 8–12 × dim)
- Early stop if no improvement for several generations
- Clamp + log-safe handling for pathological candidates

After optimization we re-run the full simulation using best parameter sets.

Next cell: implementation of a lightweight Differential Evolution routine and parameter search wrappers.

In [8]:
import math, time

# ============================================================
# Differential Evolution Optimizer (lightweight)
# ============================================================
def differential_evolution(objective, bounds, pop_factor=10, F=0.7, CR=0.9, generations=30, seed=None, tol=1e-6, stall_generations=8):
    if seed is not None:
        np.random.seed(seed)
    dim = len(bounds)
    pop_size = max(pop_factor * dim, 4)
    # Initialize population uniformly inside bounds
    pop = np.array([
        [np.random.uniform(low, high) for (low, high) in bounds]
        for _ in range(pop_size)
    ])
    scores = np.array([objective(ind) for ind in pop])
    best_idx = int(np.argmin(scores))
    best = pop[best_idx].copy()
    best_score = scores[best_idx]
    no_improve = 0
    history = [(0, best_score)]

    for gen in range(1, generations+1):
        for i in range(pop_size):
            # Mutation: select 3 distinct other indices
            idxs = [idx for idx in range(pop_size) if idx != i]
            a, b, c = pop[np.random.choice(idxs, 3, replace=False)]
            mutant = a + F * (b - c)
            # Crossover
            trial = pop[i].copy()
            j_rand = np.random.randint(0, dim)
            for j in range(dim):
                if np.random.rand() < CR or j == j_rand:
                    trial[j] = mutant[j]
            # Clamp to bounds
            for j, (low, high) in enumerate(bounds):
                if trial[j] < low: trial[j] = low
                if trial[j] > high: trial[j] = high
            # Evaluate
            trial_score = objective(trial)
            if trial_score < scores[i]:
                pop[i] = trial
                scores[i] = trial_score
                if trial_score < best_score - tol:
                    best_score = trial_score
                    best = trial.copy()
        if best_score < history[-1][1] - tol:
            no_improve = 0
        else:
            no_improve += 1
        history.append((gen, best_score))
        if no_improve >= stall_generations:
            break
    return {'best_params': best, 'best_score': best_score, 'history': history}

# ============================================================
# Objective helpers: short-horizon simulation for each filter
# ============================================================

def short_sim_prepare(common_initial_weights, num_neurons, num_features):
    # Provide fresh copies of initial weights per optimization call
    return [np.copy(w) for w in common_initial_weights]

SHORT_STEPS = 400  # reduced horizon for speed

# Control schedule reused (figure8 default). We'll precompute controls for speed.
precomputed_u = None

def ensure_precomputed_controls(dt, trajectory_type='figure8'):
    global precomputed_u
    if precomputed_u is None or len(precomputed_u) != SHORT_STEPS:
        precomputed_u = []
        for k in range(SHORT_STEPS):
            t_current = k * dt
            precomputed_u.append(generate_realistic_trajectory(t_current, trajectory_type))
        precomputed_u = np.array(precomputed_u)
    return precomputed_u

# Shared small plant wrapper for objective

def run_short_sim_EKF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias):
    Q_init, R_init, P_init, eta = params
    num_neurons = 3
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    ekf_local = EKF_RHONN_Trainer(num_neurons, num_features, initial_weights=weights_init, Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta)
    x_true = np.zeros((SHORT_STEPS, 3))
    x_hat = np.zeros((SHORT_STEPS, 3))
    # initial states already zero
    controls = ensure_precomputed_controls(dt)
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        ekf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        x_state_z = np.copy(x_hat[k])
        x_state_z[0] = x_hat[k][0]
        x_hat[k+1, 0] = RHONN_predict(x_state_z, ekf_local.weights[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, ekf_local.weights[1], u_k)
        x_hat[k+1, 2] = RHONN_predict(x_state_z, ekf_local.weights[2], u_k)
    # total MSE
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)+np.mean(err[:,2]**2)


def run_short_sim_UKF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias):
    Q_init, R_init, P_init, eta, alpha = params
    num_neurons = 3
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    ukf_local = UKF_RHONN_Trainer(num_neurons, num_features, initial_weights=weights_init, Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta, alpha=alpha, beta=2.0)
    x_true = np.zeros((SHORT_STEPS, 3))
    x_hat = np.zeros((SHORT_STEPS, 3))
    controls = ensure_precomputed_controls(dt)
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        ukf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        x_state_z = np.copy(x_hat[k])
        x_state_z[0] = x_hat[k][0]
        x_hat[k+1, 0] = RHONN_predict(x_state_z, ukf_local.weights[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, ukf_local.weights[1], u_k)
        x_hat[k+1, 2] = RHONN_predict(x_state_z, ukf_local.weights[2], u_k)
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)+np.mean(err[:,2]**2)


def run_short_sim_PF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias, n_particles_fixed=500):
    Qx, Qy, Qth, Rx, Ry, Rth, ess_ratio = params
    num_neurons = 3
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    pf_local = PF_RHONN_Trainer(num_neurons, num_features, n_particles=n_particles_fixed, initial_weights=weights_init, Q_std=[Qx,Qy,Qth], R_std=[Rx,Ry,Rth], ess_threshold=n_particles_fixed*ess_ratio)
    # Force identical initialization
    for i in range(num_neurons):
        pf_local.particles[i] = np.tile(weights_init[i], (pf_local.n_particles,1))
        pf_local.weights_pf[i] = np.ones(pf_local.n_particles)/pf_local.n_particles
    x_true = np.zeros((SHORT_STEPS, 3))
    x_hat = np.zeros((SHORT_STEPS, 3))
    controls = ensure_precomputed_controls(dt)
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        pf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        pf_w = pf_local.get_estimate()
        x_state_z = np.copy(x_hat[k])
        x_state_z[0] = x_hat[k][0]
        x_hat[k+1, 0] = RHONN_predict(x_state_z, pf_w[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, pf_w[1], u_k)
        x_hat[k+1, 2] = RHONN_predict(x_state_z, pf_w[2], u_k)
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)+np.mean(err[:,2]**2)

# ============================================================
# Wrapper objectives with logging & penalty for instability
# ============================================================

def make_objective(filter_name, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias):
    def obj(params):
        try:
            if filter_name=='EKF':
                return run_short_sim_EKF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
            elif filter_name=='UKF':
                return run_short_sim_UKF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
            else:
                return run_short_sim_PF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
        except Exception as e:
            # Heavy penalty if something blows up
            return 1e6
    return obj

# ============================================================
# Launch optimization (set RUN_DE=True to execute)
# ============================================================
RUN_DE = True  # toggle to False to skip

# Define common_initial_weights before use
import numpy as np
num_neurons = 3
num_features = 6  # Adjust this if your RHONN uses a different number of features
common_initial_weights = [np.zeros(num_features) for _ in range(num_neurons)]

if RUN_DE:
    print("\n[DE] Starting hyperparameter optimization (short horizon)...")
    start_total = time.time()
    
    # 🎲 Generate DE seed derived from main seed but different for each run
    # This ensures DE uses different optimization paths each time but remains traceable
    DE_SEED = (RANDOM_SEED + 12345) % 100000  # Different but deterministic from main seed
    print(f"🔧 DE optimization seed: {DE_SEED} (derived from main seed: {RANDOM_SEED})")

    # Define missing variables with example/default values
    dt = 0.05  # time step (seconds)
    process_noise_type = 'laplacian'  # or another type your plant() supports
    process_noise_std = 0.01  # standard deviation of process noise
    terrain_roughness = 0.0  # flat terrain by default
    sensor_bias = 0.0  # no sensor bias by default

    # Capture current initial weights snapshot for reproducibility
    initial_weights_snapshot = [np.copy(w) for w in common_initial_weights]

    # EKF
    ekf_bounds = [ (1e-6,1e-2), (1e-5,1e-1), (0.1,10.0), (0.1,1.2) ]
    ekf_obj = make_objective('EKF', initial_weights_snapshot, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
    ekf_res = differential_evolution(ekf_obj, ekf_bounds, pop_factor=10, generations=35, seed=DE_SEED)
    print(f"[DE][EKF] Best score={ekf_res['best_score']:.6e} params={ekf_res['best_params']}")

    # UKF - Same seed for consistency within this run
    ukf_bounds = [ (1e-6,1e-2), (1e-5,1e-1), (0.1,10.0), (0.1,1.2), (1e-4,0.5) ]
    ukf_obj = make_objective('UKF', initial_weights_snapshot, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
    ukf_res = differential_evolution(ukf_obj, ukf_bounds, pop_factor=10, generations=35, seed=DE_SEED)
    print(f"[DE][UKF] Best score={ukf_res['best_score']:.6e} params={ukf_res['best_params']}")

    # PF - Same seed for consistency within this run (fixed n_particles=500)
    pf_bounds = [ (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (0.3,0.9) ]
    pf_obj = make_objective('PF', initial_weights_snapshot, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
    pf_res = differential_evolution(pf_obj, pf_bounds, pop_factor=12, generations=40, seed=DE_SEED)
    print(f"[DE][PF ] Best score={pf_res['best_score']:.6e} params={pf_res['best_params']}")

    total_time = time.time()-start_total
    print(f"[DE] Optimization finished in {total_time:.1f}s")
else:
    ekf_res = ukf_res = pf_res = None

# Store for later use by main simulation rerun
optimized_params = {
    'EKF': ekf_res['best_params'] if RUN_DE else None,
    'UKF': ukf_res['best_params'] if RUN_DE else None,
    'PF' : pf_res['best_params'] if RUN_DE else None
}
print("Optimized parameter sets (raw):", optimized_params)


[DE] Starting hyperparameter optimization (short horizon)...
🔧 DE optimization seed: 71210 (derived from main seed: 58865)
[DE][EKF] Best score=1.000000e+06 params=[0.00694393 0.06297379 6.72170611 1.00084412]
[DE][UKF] Best score=1.000000e+06 params=[0.00694393 0.06297379 6.72170611 1.00084412 0.26211076]
[DE][PF ] Best score=1.000000e+06 params=[0.69466835 0.63007122 0.66919034 0.81913026 0.52460221 0.70155431
 0.49100077]
[DE] Optimization finished in 0.4s
Optimized parameter sets (raw): {'EKF': array([0.00694393, 0.06297379, 6.72170611, 1.00084412]), 'UKF': array([0.00694393, 0.06297379, 6.72170611, 1.00084412, 0.26211076]), 'PF': array([0.69466835, 0.63007122, 0.66919034, 0.81913026, 0.52460221,
       0.70155431, 0.49100077])}


# 🚀 **Section 5: Main Simulation and Training**

## **Simulation Configuration**

### **📊 Experimental Setup**
- **Duration**: 1500 time steps (30 seconds at dt=0.02s)
- **Robot trajectory**: Figure-8 pattern for comprehensive dynamics testing
- **Noise model**: Mixed process noise with terrain roughness and sensor bias
- **Initial conditions**: Robot starts at origin [0, 0, 0] 

### **🎯 Training Approach**
1. **Series-parallel identification**: Use true robot state at time k to construct features
2. **Target prediction**: Predict robot state at time k+1  
3. **Online learning**: Filters adapt RHONN weights continuously during simulation
4. **Fair comparison**: All filters start with identical initial weight vectors

### **📈 Evaluation Metrics**
- **Mean Squared Error (MSE)** for each state component (x, y, θ)
- **Total MSE** as overall performance indicator
- **Real-time learning curves** showing adaptation progress

### **⚙️ Parameter Sources**
- **EKF & UKF**: Use optimized parameters from Differential Evolution
- **PF**: Use manually tuned parameters for controlled comparison
- **Fallback**: Default parameters if optimization was skipped

## **What Happens Next**
The simulation will run all three filters simultaneously on the same robot trajectory, allowing direct performance comparison of the RHONN training approaches.

In [9]:
# ============================================================
# 5) Simulation Main Loop (uses optimized params if present)
# ============================================================

# --- Simulation settings ---
n_steps = 1500
dt = 0.02
t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

process_noise_type = 'mixed'  # 'mixed' | 'laplacian' | 'uniform' | 'gaussian'
process_noise_std = 0.01
terrain_roughness = 0.01
sensor_bias = [0.001, 0.005, 0.001]  # Small systematic biases [x, y, theta]

# --- True system init ---
x_true = np.zeros((n_steps, 3))
x_true[0] = [0.0, 0.0, 0.0]  # Initial conditions for mobile robot [x, y, theta]

# --- Control trajectory ---
trajectory_type = 'figure8'  # 'straight', 'circle', 'figure8', 'mixed', 'obstacle_avoidance'

# --- RHONN config ---
num_neurons = 3  # Three states for mobile robot [x, y, theta]
num_features = 17  # Feature vector size for 3 states + controls
num_weights_per_neuron = num_features

# --- Common initial weights for fair comparison ---
common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
print("Common Initial Weights:")
for i, w in enumerate(common_initial_weights):
    print(f"  Neuron {i}: {w}")

# Extract optimized parameters if available
opt_EKF = optimized_params.get('EKF') if 'optimized_params' in globals() else None
opt_UKF = optimized_params.get('UKF') if 'optimized_params' in globals() else None
opt_PF  = optimized_params.get('PF')  if 'optimized_params' in globals() else None

# Fallback defaults
if opt_EKF is None:
    opt_EKF = [2e-4, 8e-3, 1.5, 0.4]
if opt_UKF is None:
    opt_UKF = [2e-4, 8e-3, 1.5, 0.6, 1e-2]
if opt_PF is None:
    # Map to Qx,Qy,Qth,Rx,Ry,Rth,ess_ratio
    opt_PF = [0.05, 0.05, 0.5, 0.05, 0.075, 0.6, 0.5]

print("\nUsing parameter sets:")
print(f"EKF -> Q_init={opt_EKF[0]:.3e} R_init={opt_EKF[1]:.3e} P_init={opt_EKF[2]:.3f} eta={opt_EKF[3]:.3f}")
print(f"UKF -> Q_init={opt_UKF[0]:.3e} R_init={opt_UKF[1]:.3e} P_init={opt_UKF[2]:.3f} eta={opt_UKF[3]:.3f} alpha={opt_UKF[4]:.3e}")
print(f"PF  -> Q=[{opt_PF[0]:.3f},{opt_PF[1]:.3f},{opt_PF[2]:.3f}] R=[{opt_PF[3]:.3f},{opt_PF[4]:.3f},{opt_PF[5]:.3f}] ESS_ratio={opt_PF[6]:.2f}")

# --- EKF ---
ekf_trainer = EKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=opt_EKF[0], R_init=opt_EKF[1], P_init=opt_EKF[2], eta=opt_EKF[3]
)
x_hat_ekf = np.zeros((n_steps, 3))
x_hat_ekf[0] = x_true[0]

# --- UKF ---
ukf_trainer = UKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=opt_UKF[0], R_init=opt_UKF[1], P_init=opt_UKF[2], eta=opt_UKF[3],
    alpha=opt_UKF[4], beta=2.0
)
x_hat_ukf = np.zeros((n_steps, 3))
x_hat_ukf[0] = x_true[0]

# --- PF --- (n_particles fixed at 800)
n_particles = 800

# Q_std_per_state = opt_PF[0:3]
# R_std_per_state = opt_PF[3:6]

Q_std_per_state = [0.05, 0.05, 0.7]  # Process noise: x, y (same), theta (smaller)
R_std_per_state = [0.05, 0.075, 0.6]  # Measurement noise: x, y (same), theta (larger)

ess_threshold = n_particles * opt_PF[6]

pf_trainer = PF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=n_particles,
    initial_weights=common_initial_weights,
    Q_std=Q_std_per_state, R_std=R_std_per_state,
    ess_threshold=ess_threshold
)
# Force identical particle initialization
for i in range(num_neurons):
    pf_trainer.particles[i] = np.tile(common_initial_weights[i], (pf_trainer.n_particles,1))
    pf_trainer.weights_pf[i] = np.ones(pf_trainer.n_particles)/pf_trainer.n_particles

x_hat_pf = np.zeros((n_steps, 3))
x_hat_pf[0] = x_true[0]

print("\nStarting mobile robot simulation (optimized params)...")
for k in range(n_steps - 1):
    t_current = k * dt
    u_current = generate_realistic_trajectory(t_current, trajectory_type)
    x_true[k+1] = plant(x_true[k], u_current, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)

    # EKF
    ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ekf[k], x_hat_previous=x_hat_ekf[k], u_input=u_current)
    x_state_for_z_ekf = np.copy(x_hat_ekf[k])
    x_state_for_z_ekf[0] = x_hat_ekf[k][0]
    x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0], u_current)
    x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1], u_current)
    x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2], u_current)

    # UKF
    ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ukf[k], x_hat_previous=x_hat_ukf[k], u_input=u_current)
    x_state_for_z_ukf = np.copy(x_hat_ukf[k])
    x_state_for_z_ukf[0] = x_hat_ukf[k][0]
    x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0], u_current)
    x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1], u_current)
    x_hat_ukf[k+1, 2] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[2], u_current)

    # PF
    pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_pf[k], x_hat_previous=x_hat_pf[k], u_input=u_current)
    pf_weight_estimates = pf_trainer.get_estimate()
    x_state_for_z_pf = np.copy(x_hat_pf[k])
    x_state_for_z_pf[0] = x_hat_pf[k][0]
    x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0], u_current)
    x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1], u_current)
    x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2], u_current)

    if k % (n_steps // 10) == 0:
        print(f"Simulation progress: {k/n_steps*100:.1f}%")

print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [ 0.25686031 -0.18740747  0.27914567  0.12275979 -0.23201989  0.09028739
 -0.45079994 -0.42233246  0.48028209  0.15968051  0.18003608  0.24716189
 -0.45654871 -0.18538534  0.22900275 -0.09760697  0.03241351]
  Neuron 1: [ 0.11539938 -0.33980569  0.33623749 -0.16341305 -0.1132878  -0.44439791
  0.30483426 -0.32595251  0.2944738   0.01043655 -0.11688882 -0.2949789
  0.44770777 -0.223338    0.39224068 -0.29380428  0.19269772]
  Neuron 2: [ 0.03054579 -0.1552034   0.40045145  0.43388767 -0.37243218  0.10746292
  0.05567003  0.31381945 -0.44508027  0.47121447 -0.28172803 -0.31439339
 -0.11116531  0.26035965  0.49097783  0.43826319 -0.42539779]

Using parameter sets:
EKF -> Q_init=6.944e-03 R_init=6.297e-02 P_init=6.722 eta=1.001
UKF -> Q_init=6.944e-03 R_init=6.297e-02 P_init=6.722 eta=1.001 alpha=2.621e-01
PF  -> Q=[0.695,0.630,0.669] R=[0.819,0.525,0.702] ESS_ratio=0.49

Starting mobile robot simulation (optimized params)...
Simulation progress: 0.0%
Si

In [ ]:
# ============================================================
# 6) Results & plots for Differential Drive Mobile Robot
# ============================================================

mse_x_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
mse_y_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
mse_theta_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)

mse_x_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
mse_y_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
mse_theta_ukf = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2)

mse_x_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
mse_y_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)
mse_theta_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)

mse_total_ekf = mse_x_ekf + mse_y_ekf + mse_theta_ekf
mse_total_ukf = mse_x_ukf + mse_y_ukf + mse_theta_ukf
mse_total_pf = mse_x_pf + mse_y_pf + mse_theta_pf
mse_totals = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf}
best_filter = min(mse_totals, key=mse_totals.get)


print("🎯" + "="*65)
print(f"🏆 MEJOR FILTRO: {best_filter} (MSE total: {mse_totals[best_filter]:.6f})")
print(f"🎲 SEMILLA USADA: {RANDOM_SEED}")
print("="*67)


print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal UKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ukf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print("\n--- Performance Comparison (MSE) ---")
print(f"EKF MSE x:     {mse_x_ekf:.6f}")
print(f"EKF MSE y:     {mse_y_ekf:.6f}")
print(f"EKF MSE theta: {mse_theta_ekf:.6f}")
print(f"UKF MSE x:     {mse_x_ukf:.6f}")
print(f"UKF MSE y:     {mse_y_ukf:.6f}")
print(f"UKF MSE theta: {mse_theta_ukf:.6f}")
print(f"PF  MSE x:     {mse_x_pf:.6f}")
print(f"PF  MSE y:     {mse_y_pf:.6f}")
print(f"PF  MSE theta: {mse_theta_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'x', 'desc': 'X Position', 'y_label': 'X Position (m)', 'chi': 'χₓ (True X)', 'x': 'X (Est.)'},
    {'idx': 1, 'var': 'y', 'desc': 'Y Position', 'y_label': 'Y Position (m)', 'chi': 'χᵧ (True Y)', 'x': 'Y (Est.)'},
    {'idx': 2, 'var': 'theta', 'desc': 'Orientation', 'y_label': 'Orientation (rad)', 'chi': 'χθ (True θ)', 'x': 'θ (Est.)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines', name=state_info['chi'], line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines', name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='blue'))
    trace_ukf = go.Scatter(x=t_history, y=x_hat_ukf[:, i], mode='lines', name=f"{state_info['x']} (UKF)", line=dict(dash='dashdot', color='green'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines', name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='red'))
    fig = go.Figure([trace_plant, trace_ekf, trace_ukf, trace_pf])
    fig.update_layout(title=f'Mobile Robot RHONN Identification - {state_info["var"]}', xaxis_title='Time (s)', yaxis_title=state_info['y_label'], legend=dict(x=0, y=1, orientation='h'), font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white')
    fig.show()

error_x_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_y_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_theta_ekf = x_true[:, 2] - x_hat_ekf[:, 2]
error_x_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_y_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
error_theta_ukf = x_true[:, 2] - x_hat_ukf[:, 2]
error_x_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_y_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_theta_pf = x_true[:, 2] - x_hat_pf[:, 2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ekf, mode='lines', name=f'EKF Err X ({mse_x_ekf:.2e})', opacity=0.7, line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ukf, mode='lines', name=f'UKF Err X ({mse_x_ukf:.2e})', opacity=0.7, line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_pf, mode='lines', name=f'PF Err X ({mse_x_pf:.2e})', opacity=0.7, line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ekf, mode='lines', name=f'EKF Err Y ({mse_y_ekf:.2e})', opacity=0.7, line=dict(color='blue', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ukf, mode='lines', name=f'UKF Err Y ({mse_y_ukf:.2e})', opacity=0.7, line=dict(color='green', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_pf, mode='lines', name=f'PF Err Y ({mse_y_pf:.2e})', opacity=0.7, line=dict(color='red', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ekf, mode='lines', name=f'EKF Err θ ({mse_theta_ekf:.2e})', opacity=0.7, line=dict(color='blue', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ukf, mode='lines', name=f'UKF Err θ ({mse_theta_ukf:.2e})', opacity=0.7, line=dict(color='green', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_pf, mode='lines', name=f'PF Err θ ({mse_theta_pf:.2e})', opacity=0.7, line=dict(color='red', dash='dash')))
fig2.update_layout(title='Identification Errors (MSE values)', xaxis_title='Time (s)', yaxis_title='Error', legend=dict(x=0, y=1, orientation='h'), font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white')
fig2.show()

# 2D Trajectory plot
fig_trajectory = go.Figure()
fig_trajectory.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1], mode='lines', name='True', line=dict(color='black', width=3)))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], mode='lines', name='EKF', line=dict(color='blue', width=2, dash='dash')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], mode='lines', name='UKF', line=dict(color='green', width=2, dash='dashdot')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], mode='lines', name='PF', line=dict(color='red', width=2, dash='dot')))
fig_trajectory.add_trace(go.Scatter(x=[x_true[0, 0]], y=[x_true[0, 1]], mode='markers', name='Start', marker=dict(color='green', size=10, symbol='star')))
fig_trajectory.add_trace(go.Scatter(x=[x_true[-1, 0]], y=[x_true[-1, 1]], mode='markers', name='End', marker=dict(color='red', size=10, symbol='square')))
fig_trajectory.update_layout(title='Trajectory Comparison (Optimized Params)', xaxis_title='X (m)', yaxis_title='Y (m)', font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white', showlegend=True)
fig_trajectory.show()


# === RESUMEN DE RENDIMIENTO CON SEMILLA ===
print("\n📊 MSE Desglosado por Filtro:")
print(f"   EKF: {mse_total_ekf:.6f}  |  UKF: {mse_total_ukf:.6f}  |  PF: {mse_total_pf:.6f}")
print(f"\n💡 Para reproducir estos resultados:")
print(f"   Principal: RANDOM_SEED = {RANDOM_SEED}")
print(f"   DE Optim.: DE_SEED = {(RANDOM_SEED + 12345) % 100000}")
print(f"   (Cambia línea 8 en celda 4 para usar semilla principal)")

# --- Parameter summary ---
print("\n--- Optimized Parameter Summary ---")
print(f"EKF params: Q={ekf_trainer.Q[0][0,0]:.3e} R={ekf_trainer.R[0][0]:.3e} P0~{ekf_trainer.P[0][0,0]:.3e} eta={ekf_trainer.eta:.3f}")
print(f"UKF params: alpha={ukf_trainer.alpha:.3e} eta={ukf_trainer.eta:.3f} Qdiag={ukf_trainer.Q[0][0,0]:.3e} R={ukf_trainer.R[0][0]:.3e}")
print(f"PF params: Q_std={pf_trainer.Q_std} R_std={pf_trainer.R_std} ESS_th={pf_trainer.ess_threshold:.1f} n_particles={pf_trainer.n_particles}")

print("\nOptimization + simulation complete.")

# 58865

🎯=================================================================
🏆 MEJOR FILTRO: PF (MSE total: 0.001682)
🎲 SEMILLA USADA: 58865

Final EKF-RHONN Weights:
  Neuron 1 (x): [ 0.45018686 -0.09522625  0.0210571   0.25727155 -0.2051299  -0.04475886
 -0.19437445 -0.3690401   0.15408881  0.03207672 -0.02938985  0.51917218
 -0.58209723 -0.10807992  0.61047106 -0.04303511  0.11203202]
  Neuron 2 (y): [-0.16490718 -0.10002338  0.00783991 -0.04154927 -0.06364989 -0.28540259
  0.06220766 -0.13725041  0.05772065  0.02199874  0.00593449 -0.59248898
 -0.00494137 -0.47277204 -0.0209069   0.81236799 -0.34178469]
  Neuron 3 (theta): [ 0.06774351 -0.89852379  2.25791851  0.30184966 -0.2496532  -0.26586483
  0.02180867  0.13580294  1.08031375  0.4672505   0.1406195  -0.39324166
 -1.81598536 -0.61625601  0.2929666  -0.08901398 -1.45108746]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 1.89408812 -0.33991311 -0.10152451  0.42578012 -0.36820451  0.16024534
  0.02720447 -0.32711792  0.05774447  0.55189526  0.


📊 MSE Desglosado por Filtro:
   EKF: 0.066936  |  UKF: 0.004599  |  PF: 0.001682

💡 Para reproducir estos resultados:
   Principal: RANDOM_SEED = 58865
   DE Optim.: DE_SEED = 71210
   (Cambia línea 8 en celda 4 para usar semilla principal)

--- Optimized Parameter Summary ---
EKF params: Q=6.944e-03 R=6.297e-02 P0~1.491e+01 eta=1.001
UKF params: alpha=2.621e-01 eta=1.001 Qdiag=6.944e-03 R=6.297e-02
PF params: Q_std=[0.05, 0.05, 0.7] R_std=[0.05, 0.075, 0.6] ESS_th=392.8 n_particles=800

Optimization + simulation complete.


# 📊 **Section 6: Results Analysis and Performance Comparison**

## **Performance Metrics**

### **📈 Mean Squared Error (MSE) Analysis**
We evaluate each filter's RHONN training performance using:
- **MSE per state**: Separate error analysis for x, y, and θ components
- **Total MSE**: Sum of all state component errors for overall ranking
- **Time-series plots**: Visual tracking error evolution during training

### **🎯 What to Look For**
1. **Convergence speed**: How quickly does each filter reduce prediction errors?
2. **Steady-state performance**: Final MSE values after learning stabilizes  
3. **Robustness**: How well does each filter handle noise and disturbances?
4. **State-specific behavior**: Which filter works best for position vs. orientation?

## **Expected Results**
- **EKF**: Fast convergence, good baseline performance
- **UKF**: Better nonlinearity handling, potentially lower final MSE
- **PF**: Most flexible, may excel with proper tuning but higher computational cost

## **Visualization Components**
1. **3D trajectory plot**: Shows robot path and filter predictions in space
2. **Error time series**: Detailed error evolution for each state component  
3. **Parameter summary**: Final optimized/manual parameters used by each filter

The best performing filter will be identified based on **total MSE** across all states.

# 🎲 **Experimentos con Semillas Variables**

## **🔄 Cómo Funciona el Sistema de Semillas**

Este notebook usa **semillas variables** para explorar diferentes configuraciones aleatorias:

- **🎯 Semilla Principal (RANDOM_SEED)**: Controla pesos iniciales, ruido de simulación, partículas PF
- **🔧 Semilla DE (DE_SEED)**: Controla la optimización Differential Evolution (derivada de la principal)
- **📊 Las semillas aparecen prominentemente** al inicio y en el resumen final
- **🏆 Completamente rastreable** - puedes reproducir cualquier resultado
- **🎲 Cada ejecución explora** diferentes espacios de parámetros automáticamente

## **📝 Registro de Mejores Resultados**

**Mantén un registro manual de tus mejores ejecuciones:**

| Semilla Principal | Semilla DE | Mejor Filtro | MSE Total | Notas |
|------------------|------------|--------------|-----------|-------|
| `XXXXX` | `YYYYY` | `XXX` | `X.XXXXXX` | Descripción del resultado |
| | | | | |
| | | | | |

## **🔁 Para Reproducir un Resultado Excelente**

1. **Anota ambas semillas** del resumen final (Principal y DE)
2. **Modifica la celda 4** (configuración de seed) 
3. **Cambia** `RANDOM_SEED = int((time.time() * 1000000) % 100000)` 
4. **Por** `RANDOM_SEED = TU_SEMILLA_PRINCIPAL_FAVORITA`
5. **Ejecuta** el notebook completo

> **Nota**: La semilla DE se calculará automáticamente como `(RANDOM_SEED + 12345) % 100000`

## **🎯 Estrategia de Experimentación**

- Ejecuta varias veces para explorar el espacio de configuraciones
- Anota semillas con MSE < 0.01 (excelente rendimiento)  
- Prueba diferentes configuraciones de parámetros manuales del PF
- Compara patrones entre diferentes semillas exitosas


To download the obtained plots as images and generate a report of results, follow these steps:

---

## 📥 Downloading Plots as Images

1. **Export Plotly Figures:**
    - For each plot (`fig`, `fig2`, `fig_trajectory`), use the Plotly modebar "Download plot as a png" button (camera icon) in the top-right corner of each interactive plot.
    - Alternatively, save programmatically:

```python
fig.write_image("state_theta_comparison.png")
fig2.write_image("identification_errors.png")
fig_trajectory.write_image("trajectory_comparison.png")
```
*You may need to install `kaleido` for static image export:*
```python
!pip install -U kaleido
```

---

## 📝 Results Report

**Summary of RHONN Training Results for Differential Drive Mobile Robot**

### 1. **Mean Squared Error (MSE) per State

| Filter | MSE X        | MSE Y        | MSE θ        | Total MSE    |
|--------|--------------|--------------|--------------|--------------|
| EKF    | 3.47e-05     | 3.86e-05     | 4.40e-02     | 4.41e-02     |
| UKF    | 1.17e-04     | 1.06e-04     | 5.60e-03     | 5.83e-03     |
| PF     | 8.42e-06     | 2.23e-05     | 1.97e-03     | 2.00e-03     |

**Best overall performance:** `PF` (Particle Filter) with lowest total MSE.

---

### 2. **Optimized Parameters**

- **EKF:** Q=9.67e-03, R=5.47e-02, P₀=9.73, η=0.89
- **UKF:** Q=9.67e-03, R=5.47e-02, P₀=9.73, η=0.89, α=0.35
- **PF:** Q=[0.97, 0.55, 0.97], R=[0.72, 0.70, 0.22], ESS_ratio=0.89

---

### 3. **How to Download the Report**

- Save this markdown cell as a `.md` file or copy-paste into your preferred editor.
- Download the exported plot images from your working directory.

---

### 4. **References to Plots**

- **state_theta_comparison.png**: State θ (orientation) comparison for all filters
- **identification_errors.png**: Identification error time series for all states
- **trajectory_comparison.png**: 2D trajectory comparison (true vs. filter predictions)

---

**End of Report**